<a href="https://colab.research.google.com/github/gitanisa008/gee_scripts/blob/main/multi_sensor_deforestation_monitor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-Sensor Deforestation Monitoring Tool
**Portfolio Version | Optical + Radar Fusion Approach**

| Item | Detail |
|---|---|
| Reference baseline | December 31, 2020 (aligned with EUDR) |
| Optical sensors | Hansen GFC, GLAD Alerts (Landsat/Sentinel-2), JRC TMF |
| Radar sensor | RADD Alerts (Sentinel-1 SAR, Wageningen University) |
| Study area | Indonesia |

---


## 0. Install Dependencies

In [ ]:
# Run once; restart kernel afterward if needed
!pip install earthengine-api geemap --quiet

## 1. Authentication and Imports

In [ ]:
import ee
import geemap
import json

# Authenticate with your Google Earth Engine account
# First time: a browser window will open
ee.Authenticate()
ee.Initialize(project='your-gee-project-id')  # Replace with your GEE project ID

## 2. Dataset Initialization

Load all source datasets:
- **JRC Global Forest Cover 2020** - forest baseline
- **JRC Tropical Moist Forest (TMF)** - deforestation 2021-2025
- **GLAD Alerts 2025** - optical change detection (Landsat/Sentinel-2)
- **RADD Alerts** - Sentinel-1 SAR radar alerts (cloud-penetrating)


In [ ]:
# Study area: Indonesia
idn = ee.FeatureCollection("FAO/GAUL/2015/level1")         .filter(ee.Filter.eq('ADM0_NAME', 'Indonesia'))

# ---- JRC Global Forest Cover 2020 ----
jrc        = ee.Image("JRC/GFC2020/V3")
forest_jrc = jrc.gt(0).clip(idn)

# ---- JRC Tropical Moist Forest: Deforestation 2021-2025 ----
tmf      = ee.ImageCollection("projects/JRC/TMF/v1_2025/DeforestationYear").mosaic()
loss_jrc = tmf.gte(2021).And(tmf.lte(2025)).clip(idn)

# ---- GLAD Optical Alerts 2025 (Landsat / Sentinel-2) ----
glad        = ee.ImageCollection('projects/glad/alert/UpdResult')                 .map(lambda img: img.select(['conf25', 'alertDate25']))
glad_latest = glad.mosaic()
glad_alerts = (
    glad_latest.select('conf25').gt(0)
    .And(glad_latest.select('alertDate25').gt(0))
)

# ---- RADD Alerts (Sentinel-1 SAR | Wageningen University) ----
radd_col  = (
    ee.ImageCollection('projects/radar-wur/raddalert/v1')
    .filter(ee.Filter.listContains('system:band_names', 'Alert'))
    .filter(ee.Filter.listContains('system:band_names', 'Date'))
)
radd_img  = radd_col.select(['Alert', 'Date']).mosaic()
radd_conf = radd_img.select('Alert')
radd_date = radd_img.select('Date')

# Post-2020 cutoff (RADD day-offset > 731 corresponds to after Dec 31 2020)
radd_post2020  = radd_date.gt(731)
radd_all_conf  = radd_post2020.And(radd_conf.gte(2))   # all confidence (>=2)
radd_high_conf = radd_post2020.And(radd_conf.gte(3))   # high confidence only (>=3)

print("Datasets loaded successfully.")

## 3. Multi-Sensor Fusion Logic

Cross RADD radar alerts with forest loss masks from JRC and GLAD to reduce false positives.


In [ ]:
# RADD x JRC Loss
radd_jrc       = radd_all_conf.multiply(loss_jrc).unmask(0).rename('radd_jrc')
radd_jrc_high  = radd_high_conf.multiply(loss_jrc).unmask(0).rename('radd_jrc_high')

# RADD x GLAD Alerts
radd_glad      = radd_all_conf.multiply(glad_alerts).unmask(0).rename('radd_glad')
radd_glad_high = radd_high_conf.multiply(glad_alerts).unmask(0).rename('radd_glad_high')

print("Fusion layers ready.")

## 4. Visualization Parameters

In [ ]:
VIS = {
    'forest_jrc': {'min': 0, 'max': 1, 'palette': ['#66bd63']},
    'loss_jrc':   {'min': 0, 'max': 1, 'palette': ['#f46d43']},
    'glad_all':   {'min': 0, 'max': 1, 'palette': ['#8B0000']},
    'radd_all':   {'min': 0, 'max': 1, 'palette': ['#fdae61']},
    'radd_high':  {'min': 0, 'max': 1, 'palette': ['#f46d43']},
}

## 5. Interactive Map

Default view centers on Indonesia. Toggle layers using the layer panel (top-right of map).


In [ ]:
Map = geemap.Map()
Map.centerObject(ee.Geometry.Rectangle([95, -11, 141, 6]), 5)

# Forest baseline
Map.addLayer(forest_jrc.updateMask(forest_jrc),
             VIS['forest_jrc'], 'JRC Forest 2020', True)

# JRC deforestation
Map.addLayer(loss_jrc.updateMask(loss_jrc.gt(0)),
             VIS['loss_jrc'], 'JRC Deforestation 2021-2025', True)

# GLAD optical alerts
Map.addLayer(glad_alerts.updateMask(glad_alerts.gt(0)),
             VIS['glad_all'], 'GLAD Alert 2025', True)

# RADD x JRC (all confidence)
Map.addLayer(radd_jrc.updateMask(radd_jrc.gt(0)),
             VIS['radd_all'], 'RADD x JRC Loss (All confidence)', False)

# RADD x GLAD (all confidence)
Map.addLayer(radd_glad.updateMask(radd_glad.gt(0)),
             VIS['radd_all'], 'RADD x GLAD Alert 2025 (All confidence)', False)

# RADD x JRC (high confidence)
Map.addLayer(radd_jrc_high.updateMask(radd_jrc_high.gt(0)),
             VIS['radd_high'], 'RADD x JRC Loss (High confidence)', False)

# RADD x GLAD (high confidence)
Map.addLayer(radd_glad_high.updateMask(radd_glad_high.gt(0)),
             VIS['radd_high'], 'RADD x GLAD Alert 2025 (High confidence)', False)

Map

## 6. Load Plot GeoJSON

Paste your GeoJSON FeatureCollection string, or use the sample below.
Expected properties per feature: `ProductionPlace` (string), `Area` (hectares).


In [ ]:
SAMPLE_GEOJSON = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {"ProductionPlace": "PLOT-001", "Area": 0.81},
            "geometry": {
                "type": "Polygon",
                "coordinates": [[[104.0, -1.0], [104.01, -1.0],
                                  [104.01, -1.01], [104.0, -1.01],
                                  [104.0, -1.0]]]
            }
        }
    ]
}

# Replace with your own GeoJSON dict or load from file:
# with open('your_plots.geojson') as f:
#     geojson_data = json.load(f)

geojson_data = SAMPLE_GEOJSON
print(f"Loaded {len(geojson_data['features'])} plot(s).")

### 6a. Display Plots on Map

In [ ]:
features = []
for feat in geojson_data['features']:
    geom  = ee.Geometry(feat['geometry'])
    props = feat.get('properties', {})
    features.append(ee.Feature(geom, props))

fc_plots = ee.FeatureCollection(features)

Map.addLayer(fc_plots, {'color': '#0288d1'}, 'Uploaded Plots')
Map.centerObject(fc_plots, 12)
print(f"{len(features)} plot(s) added to map.")
Map

## 7. Per-Plot Alert Analysis

Runs `reduceRegion` for each plot across all three sensors at 30 m resolution.
Alert level logic:
- 3 sensors triggered -> HIGH
- 2 sensors triggered -> MEDIUM
- 1 sensor triggered -> LOW
- 0 sensors triggered -> NO ALERT


In [ ]:
def compute_alert(ee_geom):
    """
    Returns (alert_level: str, triggered_sensors: list[str])
    Uses server-side reduceRegion; blocks until result is evaluated.
    """
    combined = ee.Image.cat([
        glad_alerts.rename('glad_s2'),
        loss_jrc.rename('jrc_loss'),
        radd_all_conf.rename('radd')
    ])

    result = combined.reduceRegion(
        reducer=ee.Reducer.max(),
        geometry=ee_geom,
        scale=30,
        maxPixels=1e6
    ).getInfo()

    has_glad = result.get('glad_s2') is not None and result['glad_s2'] > 0
    has_jrc  = result.get('jrc_loss') is not None and result['jrc_loss'] > 0
    has_radd = result.get('radd')     is not None and result['radd']     > 0

    sensor_count = sum([has_glad, has_jrc, has_radd])

    level_map = {3: 'HIGH', 2: 'MEDIUM', 1: 'LOW', 0: 'NO ALERT'}
    level     = level_map[sensor_count]

    triggered = (
        (['GLAD-S2'] if has_glad else []) +
        (['JRC']     if has_jrc  else []) +
        (['RADD']    if has_radd else [])
    )
    return level, triggered


print("compute_alert() ready.")

### 7a. Run Analysis for All Loaded Plots

In [ ]:
import pandas as pd

results = []
for feat in geojson_data['features']:
    props   = feat.get('properties', {})
    ee_geom = ee.Geometry(feat['geometry'])
    level, triggered = compute_alert(ee_geom)
    results.append({
        'ProductionPlace': props.get('ProductionPlace', 'N/A'),
        'Area_ha':         props.get('Area', 'N/A'),
        'AlertLevel':      level,
        'TriggeredSensors': ', '.join(triggered) if triggered else 'None'
    })

df = pd.DataFrame(results)
df

### 7b. Color-Coded Summary Table

In [ ]:
COLOR_MAP = {
    'HIGH':     'background-color: #ffcdd2; color: #b71c1c; font-weight: bold',
    'MEDIUM':   'background-color: #ffe0b2; color: #e65100; font-weight: bold',
    'LOW':      'background-color: #fff9c4; color: #f9a825; font-weight: bold',
    'NO ALERT': 'background-color: #c8e6c9; color: #2e7d32; font-weight: bold',
}

def style_alert(val):
    return COLOR_MAP.get(val, '')

df.style.applymap(style_alert, subset=['AlertLevel'])

## 8. Export Results to CSV

In [ ]:
output_path = 'deforestation_alert_results.csv'
df.to_csv(output_path, index=False)
print(f"Results saved to {output_path}")
df.to_csv(output_path, index=False)

## 9. Optional: Export Alert Raster to Google Drive

Export one of the fusion layers as a GeoTIFF to your Google Drive.


In [ ]:
# Example: export RADD x JRC (high confidence) clipped to uploaded plots extent
export_task = ee.batch.Export.image.toDrive(
    image=radd_jrc_high.updateMask(radd_jrc_high.gt(0)),
    description='RADD_JRC_HighConf_Export',
    folder='GEE_Exports',
    fileNamePrefix='radd_jrc_high_conf',
    region=fc_plots.geometry().bounds(),
    scale=30,
    crs='EPSG:4326',
    maxPixels=1e9
)

export_task.start()
print("Export task submitted. Check your GEE Tasks panel or Google Drive.")
print("Task ID:", export_task.id)